# Benchmark: manuálne spracovanie vs. automatizácia v jazyku Python

Notebook reprodukuje metodiku **Teixeira, Ferreira a Ramos (2024)** v zmenšenej podobe na slovenskom datasete účtovných závierok 2008. Pre štyri reprezentatívne analytické úlohy sa porovnáva

- **predpokladaný manuálny čas** (analytik v Exceli s priemernými znalosťami) a
- **automatizovaný čas** v jazyku Python (medián z N opakovaných meraní pomocou `time.perf_counter`).

Manuálne časy sú konzervatívne odhady na základe počtu krokov vyžadovaných v MS Exceli (otvorenie, filtrovanie, agregácia, kopírovanie). Slúžia ako referenčný bod pre poriadok veľkosti, nie ako empiricky namerané hodnoty. Pre presnejšie hodnoty by bolo potrebné vykonať časomieru priamo na používateľoch (mimo rámec bakalárskej práce).

Výstupná tabuľka je formátovaná podľa vzoru tabuľky 9 v texte práce.

In [1]:
import sys, time
from pathlib import Path

# umožní spúšťať notebook z priečinka notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.load import load_clean_dataset
from src.ratios import compute_ratios
from src.benchmark import compare_to_industry, industry_aggregates

df = load_clean_dataset()
ratios = compute_ratios(df)
print(f'Dataset: {len(df)} firiem, {df.shape[1]} stĺpcov')

Dataset: 246 firiem, 557 stĺpcov


In [2]:
def time_call(fn, repeat: int = 50):
    """Vráti medián doby vykonávania `fn()` v sekundách z `repeat` spustení."""
    timings = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        timings.append(time.perf_counter() - t0)
    return sorted(timings)[len(timings) // 2]

## Úloha A — Filtrácia a agregácia

**Zadanie**: nájsť všetky firmy v Bratislavskom kraji s tržbami > 100 mil. SKK a vypočítať ich priemernú ROA.

**Manuálny postup v Exceli**: otvoriť súbor (~10 s), aplikovať filter na stĺpec *Kraj* (~15 s), aplikovať filter na stĺpec *Tržby (spolu)* > 100 000 000 (~25 s), pre každý riadok vypočítať ROA = výsledok hospodárenia / spolu majetok (~60 s), priemer cez funkciu AVERAGE (~10 s). **Spolu: ~120 s.**

In [3]:
def task_a():
    sub = df[(df['Kraj'] == 'Bratislavský') & (df['Tržby (spolu)'] > 100_000_000)]
    matched = ratios[ratios['Ico'].isin(sub['Ico'])]
    return matched['prof_roa'].mean()

result_a = task_a()
time_a = time_call(task_a)
print(f'Priemerná ROA: {result_a:.2%}')
print(f'Automatizovaný čas (medián): {time_a*1000:.2f} ms')

Priemerná ROA: -0.69%
Automatizovaný čas (medián): 3.34 ms


## Úloha B — Generovanie benchmark-reportu pre konkrétnu firmu

**Zadanie**: pre firmu TESCO STORES SR (IČO 31321828) zostaviť porovnávaciu tabuľku všetkých kľúčových ukazovateľov voči mediánu odvetvia *Maloobchod*.

**Manuálny postup v Exceli**: otvoriť súbor a nájsť firmu (~30 s), filtrovať odvetvie *Maloobchod* (~15 s), pre 6 ukazovateľov vypočítať medián vzorky (~6 × 30 s = 180 s), kopírovať hodnoty firmy do porovnávacej tabuľky (~60 s), porovnať a označiť pozíciu (~30 s). **Spolu: ~315 s.**

In [4]:
def task_b():
    return compare_to_industry(ratios, '31321828')

result_b = task_b()
time_b = time_call(task_b, repeat=20)
print(result_b.to_string(index=False))
print(f'\nAutomatizovaný čas (medián): {time_b*1000:.2f} ms')

        ukazovateľ  firma  medián odvetvia    Q25    Q75  n_peers                    pozícia
          liq_cash 0.1445           0.1489 0.0402 0.3635        5 v medzikvartilovom rozsahu
       liq_current 0.2203           1.2684 1.0894 1.7496        5           pod 1. kvartilom
  dbt_equity_ratio 0.6238           0.1316 0.1062 0.4112        5           nad 3. kvartilom
prof_ebitda_margin 0.0660           0.0789 0.0335 0.1200        5 v medzikvartilovom rozsahu
act_asset_turnover 1.2491           1.1927 0.0000 3.2531        8 v medzikvartilovom rozsahu
  altman_z_finstat 2.0416           4.3763 2.5936 4.7435        5           pod 1. kvartilom

Automatizovaný čas (medián): 7.23 ms


## Úloha C — Detekcia firiem v finančnom distrese

**Zadanie**: identifikovať všetky firmy s Altman Z-skóre < 1.81 (zóna distresu) a zoradiť ich podľa kraja.

**Manuálny postup v Exceli**: otvoriť súbor (~10 s), nájsť stĺpec Altman Z-skóre (~15 s), aplikovať filter < 1.81 (~20 s), pridať triedenie podľa kraja (~30 s), preformátovať a uložiť (~30 s). **Spolu: ~105 s.**

In [5]:
def task_c():
    distress = ratios[ratios['risk_altman']]
    return distress.sort_values('Kraj')[['Nazov', 'Kraj', 'Odvetvie', 'altman_z_own']]

result_c = task_c()
time_c = time_call(task_c)
print(f'Firiem v zóne distresu: {len(result_c)}')
print(result_c.head(10).to_string(index=False))
print(f'\nAutomatizovaný čas (medián): {time_c*1000:.2f} ms')

Firiem v zóne distresu: 34
                                             Nazov            Kraj                       Odvetvie  altman_z_own
                                          HKM a.s. Banskobystrický         Cestovný ruch a gastro      1.540899
                          LUKA & BRAMER GROUP a.s.    Bratislavský                         Služby      0.059458
Mercedes-Benz Financial Services Slovakia s. r. o.    Bratislavský                       Financie      1.362567
           BURDA PRINT CEE s. r. o. - v likvidácii    Bratislavský Média, vydavateľstvá a kultúra      1.415693
                                        ENFER a.s.    Bratislavský                  Nehnuteľnosti     -0.702690
              VENTURE PARTNERS, a. s. v likvidácii    Bratislavský                  Nehnuteľnosti     -0.236934
                   CEVA Logistics Slovakia, s.r.o.    Bratislavský            Doprava a logistika      0.813120
                                      DENIM s.r.o.    Bratislavský           

## Úloha D — Odvetvové mediány pre celý dataset

**Zadanie**: vypočítať medián siedmich kľúčových ukazovateľov pre každé z 32 odvetví (kontingenčná tabuľka 32×7).

**Manuálny postup v Exceli**: otvoriť súbor (~10 s), zostaviť kontingenčnú tabuľku (~120 s), pridať polia hodnôt s funkciou MEDIAN (~7 × 30 s = 210 s), formátovať tabuľku (~60 s), exportovať (~30 s). **Spolu: ~430 s.**

In [6]:
def task_d():
    return industry_aggregates(ratios)

result_d = task_d()
time_d = time_call(task_d, repeat=20)
print(f'Odvetví: {len(result_d)}')
print(f'Automatizovaný čas (medián): {time_d*1000:.2f} ms')

Odvetví: 32
Automatizovaný čas (medián): 16.82 ms


## Súhrnná tabuľka výsledkov (vzor: tabuľka 9 v texte práce)

In [7]:
manual_times = {
    'A': 120,
    'B': 315,
    'C': 105,
    'D': 430,
}
auto_times = {
    'A': time_a,
    'B': time_b,
    'C': time_c,
    'D': time_d,
}
labels = {
    'A': 'Filtrácia firiem v Bratislavskom kraji s tržbami > 100 mil. a výpočet priemerného ROA',
    'B': 'Benchmark report pre konkrétnu firmu vs. odvetvový medián (TESCO/Maloobchod)',
    'C': 'Detekcia firiem v finančnom distrese (Altman Z < 1.81), zoradených podľa kraja',
    'D': 'Výpočet odvetvových mediánov siedmich ukazovateľov pre 32 odvetví',
}
rows = []
for k in 'ABCD':
    m = manual_times[k]
    a = auto_times[k]
    rows.append({
        'Úloha':                labels[k],
        'Manuálne (s)':         m,
        'Automatizované (s)':   round(a, 4),
        'Úspora času (%)':      round(100 * (m - a) / m, 2),
        'Faktor zrýchlenia':    f'{m / a:.0f}×',
    })
summary = pd.DataFrame(rows)
summary.to_csv('../reports/output/benchmark_summary.csv', index=False)
summary

,Úloha,Manuálne (s),Automatizované (s),Úspora času (%),Faktor zrýchlenia
0,Filtrácia firiem v Bratislavskom kraji s tržba...,120,0.0033,100.0,35967×
1,Benchmark report pre konkrétnu firmu vs. odvet...,315,0.0072,100.0,43581×
2,Detekcia firiem v finančnom distrese (Altman Z...,105,0.0016,100.0,64835×
3,Výpočet odvetvových mediánov siedmich ukazovat...,430,0.0168,100.0,25566×


In [8]:
avg_saving = summary['Úspora času (%)'].mean()
total_manual = summary['Manuálne (s)'].sum()
total_auto = summary['Automatizované (s)'].sum()
print(f'Priemerná úspora času: {avg_saving:.2f} %')
print(f'Celkový manuálny čas: {total_manual} s')
print(f'Celkový automatizovaný čas: {total_auto:.4f} s')
print(f'Faktor celkového zrýchlenia: {total_manual / total_auto:.0f}×')

Priemerná úspora času: 100.00 %
Celkový manuálny čas: 970 s
Celkový automatizovaný čas: 0.0289 s
Faktor celkového zrýchlenia: 33564×


## Interpretácia

Výsledky sú v súlade so zisteniami **Teixeira, Ferreira a Ramos (2024)**, ktorí na dvoch úlohách dokumentovali zníženie z 583 s a 302 s na ~7 s, t.j. úsporu času okolo 96 %.

Limitácie merania:

1. **Manuálne časy sú odhady, nie merané hodnoty.** Pre rigorózne porovnanie by bolo potrebné experimentálne meranie na vzorke používateľov s rôznou úrovňou znalostí Excelu.
2. **Dataset je relatívne malý** (246 firiem). Pre väčšie datasety by sa rozdiel ďalej zväčšil — automatizovaný čas rastie s `n` lineárne, manuálny prevažne kvadraticky (väčšie filtre, viac kopírovania).
3. **Zahrnutý je iba čas vykonávania úlohy**, nie čas vývoja skriptu. Investícia do vývoja sa však amortizuje pri opakovanom použití (Ylä-Kujala et al., 2023, podkapitola 1.2.4).